In [ ]:
source("Main.R")
source("Conf.R")
source("Utilities.R")
library("factoextra")
library("maptree")
library(repr)
library("ggpubr")

In [ ]:
selCoefs <- read.csv(referencePath("SignificantCoefMatrix.csv"), row.names = 1)
selCoefs <- t(selCoefs)
KOGenes <-  colnames(selCoefs)
KOGenes <- sapply(KOGenes, function(x){strsplit(x,"_")[[1]][2]})
colnames(selCoefs) <- KOGenes

In [ ]:
geneModules <- read.csv(referencePath("ME_GeneModules_leiden_11_Modules.csv"), row.names = 1)
guideModules <- read.csv(referencePath("ME_GuideModules_leiden_6_Modules.csv"), row.names = 1)


In [ ]:
geneModules$GeneName[geneModules$GeneName %ni% rownames(selCoefs)]

In [ ]:
dim(selCoefs)

In [ ]:
guideModules$GuideName[guideModules$GuideName %ni% colnames(selCoefs)]

In [ ]:
selCoefs <- selCoefs[geneModules$GeneName,guideModules$GuideName]

In [ ]:
res = read.csv(file.path("TextFiles", "KOModuleEffects.csv"), row.names = 1)

In [ ]:
res = res[res$term %ni% c("(Intercept)", "n_genes", "mt_frac", paste0("leiden",1:9)),]

In [ ]:
head(res)

In [ ]:
coefMat = data.frame(reshape(res[,c("term","estimate","respGene")], idvar = "term",
                  timevar = "respGene", direction="wide"))

pValMat = data.frame(reshape(res[,c("term","p.value","respGene")], idvar = "term",
                  timevar = "respGene", direction="wide"))

rownames(coefMat) = coefMat$term
rownames(pValMat) = pValMat$term
coefMat$term = NULL
pValMat$term = NULL

colnames(coefMat) <- sapply(colnames(coefMat), function(x){strsplit(x,"estimate.")[[1]][2]})
colnames(pValMat) <- sapply(colnames(pValMat), function(x){strsplit(x,"p.value.")[[1]][2]})

In [ ]:
head(coefMat)

In [ ]:
geneModules$GeneName[geneModules$GeneName %ni% colnames(coefMat) ]

In [ ]:
colnames(coefMat)[colnames(coefMat) %ni% geneModules$GeneName ]

In [ ]:
coefMat['X0610012G03Rik'] = coefMat[,'0610012G03Rik']
coefMat['X2010005H15Rik'] = coefMat[,'2010005H15Rik']
coefMat['X2010111I01Rik'] = coefMat[,'2010111I01Rik']
coefMat['X2310001H17Rik'] = coefMat[,'2310001H17Rik']
coefMat['X2810474O19Rik'] = coefMat[,'2810474O19Rik']
coefMat[,c('0610012G03Rik','2010005H15Rik',
           '2010111I01Rik', '2310001H17Rik','2810474O19Rik')] = NULL

In [ ]:
colnames(coefMat)[colnames(coefMat) %ni% geneModules$GeneName ]

In [ ]:
coefMat = t(coefMat)
coefMat = coefMat[geneModules$GeneName,]

In [ ]:
guideModules$GuideGroup = paste0("K_",guideModules$GuideGroup)

In [ ]:
allRes= list()

for( i in colnames(coefMat)){

    for(j in colnames(selCoefs)){
    
        if(guideModules[j,"GuideGroup"] == i){
            myR2 = computeR2(selCoefs[,j], coefMat[,i])
            pearsonCor = round(cor(selCoefs[,j], coefMat[,i], method = "pearson"), digits = 2)

            allRes = lappend(allRes, c(KOModule=i,KOGene = j, myR2= myR2,pearsonCor =pearsonCor,
                                       "moduleGene"=TRUE))
            
        }else{
            myR2 = computeR2(selCoefs[,j], coefMat[,i])
            pearsonCor = round(cor(selCoefs[,j], coefMat[,i], method = "pearson"), digits = 2)

            allRes = lappend(allRes, c(KOModule=i,KOGene = j, myR2= myR2,pearsonCor =pearsonCor,
                                       "moduleGene"=FALSE))
        }
    
    }

}

allRes <- data.frame(do.call(rbind,allRes))

In [ ]:
head(allRes)

allRes[allRes$KOModule == "K_0","KOModule"] = "M2"
allRes[allRes$KOModule == "K_1","KOModule"] = "M3"
allRes[allRes$KOModule == "K_2","KOModule"] = "M6"
allRes[allRes$KOModule == "K_3","KOModule"] = "M5"
allRes[allRes$KOModule == "K_4","KOModule"] = "M1"
allRes[allRes$KOModule == "K_5","KOModule"] = "M4"

In [ ]:
allRes[allRes$moduleGene == FALSE,"KOModule"] = "Other module"
allRes$KOModule = factor(allRes$KOModule, levels=c("Other module", "M1","M2","M3","M4","M5","M6"))
allRes$pearsonCor <- as.numeric(allRes$pearsonCor)

In [ ]:
allRes

In [ ]:
options(repr.plot.width=10, repr.plot.height=6)
#allResTrue = allRes[allRes$moduleGene == TRUE,]

allResTrue = allRes
give.n <- function(x){
  return(c(y = mean(x)*1.1, label = length(x))) 
  # experiment with the multiplier to find the perfect position
}

xx <- ggplot(allResTrue, aes(x=KOModule, y=pearsonCor, fill=KOModule)) + 
  geom_boxplot(outlier.colour="red", outlier.shape=8,outlier.size=4)+
  scale_fill_manual(values=c( "yellow",
                              "#1f77b4",
                              "#ff7f0e",
                              "#279e68",
                              "#d62728", 
                              "#aa40fc", 
                              "#8c564b"
                               )) +xlab("KO Module")+  theme_minimal()+
     theme(axis.text.x = element_text( size=15),
        axis.text.y = element_text( size=15),
        axis.title= element_text( size=15)
       )+
       stat_compare_means(method="t.test",paired = FALSE,
                                comparisons = list(c("Other module", "M1"),
                                                   c("Other module", "M2"),
                                                   c("Other module", "M3"),
                                                   c("Other module", "M4"),
                                                   c("Other module", "M5"),
                                                   c("Other module", "M6")),
                                color="red", method.args = list(alternative = "less"),
                                label.y = c(1,1.2,1.4,1.6,1.8,2.0), aes(label = ..p.signif..))+
ylab("Pearson correlation between \n overall module effect with individual KO effects")+
stat_summary(fun.data = give.n, geom = "text", fun.y = median)

ggsave("figures/FigS3N.pdf", width = 12, height = 6)



In [ ]:
allResTrue$myR2 = NULL
allResTrue$moduleGene = NULL

In [ ]:
head(allResTrue)

In [ ]:
write.csv(allResTrue,"outputs/ModuleGenesExplained.csv",quote=FALSE, row.names=FALSE)

In [ ]:
k <- read.csv(textfilesPath("ME_LMBetaFDRALL.csv.gz"), row.names = 1)

In [ ]:
k = k[,geneModules$GeneName]

In [ ]:
str(k)

In [ ]:
nSignGenes = apply(k, 1, function(x){1041 - length(x[x > 0.1])})
nSignGenes = data.frame("NumSignCond"= nSignGenes,"GeneName" = names(nSignGenes) )

In [ ]:
nSignGenes

In [ ]:
nSignGenes["GENE_Rnf128_",]